In [1]:
!pip install torch torch_geometric scikit-learn pandas matplotlib --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.1 MB/s eta 0:00:00


**Import Data**

In [2]:
#Utils
import torch
from torch_geometric.data import Data
import pandas as pd
#For model making
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch.nn import Linear
# For pre-processing and manupilation
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Sistem Rekomendasi/Mobile-Price-Prediction-cleaned_data.csv')
features = df.iloc[:, :8] # select first 8 columns or specify explicitly
# Normalize
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
x = torch.tensor(features_scaled, dtype=torch.float)

In [4]:
# Similarity
similarity_matrix = cosine_similarity(features_scaled)
edge_index = []
threshold = 0.9 #higher = fewer but stronger connections
for i in range(len(df)):
    for j in range(len(df)):
        if i != j and similarity_matrix[i][j] > threshold:
            edge_index.append([i, j])
edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
data = Data(x=x, edge_index=edge_index) #This is our data in graph form now (via Tensor Geometery)

In [5]:
# Model Building
class PhoneGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=32):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels) #uses aggreationa and summary
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.lin = Linear(hidden_channels, hidden_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return self.lin(x)

In [6]:
# Training
model = PhoneGNN(in_channels=x.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
model.train()
for epoch in range(1000):
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = ((out @ out.T) - torch.eye(len(x))).pow(2).mean() #loss function (details I have put below)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.0145
Epoch 10, Loss: 0.0182
Epoch 20, Loss: 0.0056
Epoch 30, Loss: 0.0034
Epoch 40, Loss: 0.0019
Epoch 50, Loss: 0.0017
Epoch 60, Loss: 0.0015
Epoch 70, Loss: 0.0015
Epoch 80, Loss: 0.0014
Epoch 90, Loss: 0.0014
Epoch 100, Loss: 0.0014
Epoch 110, Loss: 0.0013
Epoch 120, Loss: 0.0013
Epoch 130, Loss: 0.0013
Epoch 140, Loss: 0.0013
Epoch 150, Loss: 0.0013
Epoch 160, Loss: 0.0013
Epoch 170, Loss: 0.0013
Epoch 180, Loss: 0.0013
Epoch 190, Loss: 0.0013
Epoch 200, Loss: 0.0013
Epoch 210, Loss: 0.0013
Epoch 220, Loss: 0.0013
Epoch 230, Loss: 0.0013
Epoch 240, Loss: 0.0013
Epoch 250, Loss: 0.0012
Epoch 260, Loss: 0.0012
Epoch 270, Loss: 0.0012
Epoch 280, Loss: 0.0012
Epoch 290, Loss: 0.0012
Epoch 300, Loss: 0.0012
Epoch 310, Loss: 0.0012
Epoch 320, Loss: 0.0012
Epoch 330, Loss: 0.0012
Epoch 340, Loss: 0.0012
Epoch 350, Loss: 0.0012
Epoch 360, Loss: 0.0012
Epoch 370, Loss: 0.0012
Epoch 380, Loss: 0.0012
Epoch 390, Loss: 0.0012
Epoch 400, Loss: 0.0012
Epoch 410, Loss: 0.0012
Epo

In [7]:
# Evaluation
model.eval()
embeddings = model(data.x, data.edge_index).detach()
query_input = [[4.6, 8, 128, 6.5, 48, 16, 5000, 30000]] # QUERY
query_scaled = scaler.transform(query_input)
query_tensor = torch.tensor(query_scaled, dtype=torch.float)
# Use the trained model WITHOUT passing edge_index (just a simple MLP pass)
with torch.no_grad():
    x_q = model.conv1.lin_l(query_tensor) # skip aggregation; use .lin_l to mimic GraphSAGE MLP
    x_q = F.relu(x_q)
    x_q = model.conv2.lin_l(x_q)
    x_q = F.relu(x_q)
    query_embedding = model.lin(x_q)
similarity_scores = cosine_similarity(query_embedding, embeddings) # Compute cosine similarity to all graph node embeddings
# Convert back to tensor (if needed)
similarity_scores = torch.tensor(similarity_scores)
top_k = torch.topk(similarity_scores, k=5)
print("Top 5 similar phone indices:", top_k.indices.tolist())

Top 5 similar phone indices: [[149, 444, 382, 112, 502]]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [8]:
# Final Results
df = pd.read_csv('/content/drive/MyDrive/Sistem Rekomendasi/Mobile-Price-Prediction-cleaned_data.csv')
top_k_indices = top_k.indices.squeeze().tolist()
recommended_phones = df.iloc[top_k_indices]
recommended_phones

,Ratings,RAM,ROM,Mobile_Size,Primary_Cam,Selfi_Cam,Battery_Power,Price
149,4.4,6.0,128.0,6.50,40,2.0,3750,13699
444,3.9,6.0,32.0,4.50,48,11.0,3000,900
382,4.0,6.0,32.0,4.50,48,12.0,3000,949
112,4.1,6.0,25.0,4.54,48,12.0,3000,1369
502,4.0,6.0,32.0,4.50,48,13.0,3000,847
